In [ ]:
import pandas as pd
import numpy as np
import random
import tensorflow as tf

print(tf.__version__)

In [ ]:
raw_dataset = pd.read_csv('dest_1005_eta_port_ataavail_1.csv')
raw_dataset.head()

In [ ]:
aisdf1 = raw_dataset.drop(['lat','lon','latlng1','ata','mmsi','dest1','navi'],axis=1)
print(aisdf1.head())
len(aisdf1['timedif(ata-utc)'].unique())

In [ ]:
timedif = aisdf1['timedif(ata-utc)']

y_data = timedif

In [ ]:
aisdf1 = aisdf1.drop('timedif(ata-utc)', axis=1)

X_data = aisdf1

In [ ]:
# 피처 스케일링
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_data_scaled = scaler.fit_transform(X_data)

X_data_scaled[0]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_data_scaled, y_data, test_size=0.2, shuffle=True, random_state = 777)

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

def build_model(num_input=1):
    model = Sequential()
    model.add(Dense(128, activation='relu', input_dim=num_input))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='linear'))
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    
    return model

In [ ]:
model = build_model(num_input=15)

In [ ]:
model.fit(X_train, y_train, epochs=10000, batch_size=32, verbose=2)
# model.fit(X_train, y_train, epochs=1000, batch_size=32, verbose=2)

In [ ]:
from sklearn.metrics import mean_absolute_error

predict = model.predict(X_test)

mae = mean_absolute_error(y_test, predict)
print("mae:", mae)

# def MAE(y_test, y_pred):
#     return np.mean(np.abs((y_test-y_pred)))

# print(MAE(y_test, predict), mae)

In [ ]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_test, predict)
print("mse:", mse)

In [ ]:
# def MAPE(y_test, y_pred): 
#     return np.mean(np.abs((y_test - y_pred) / y_test)) * 100
# y_pred = model.predict(X_test)
# MAPE(y_test, y_pred)

In [ ]:
model.evaluate(X_test, y_test)

In [ ]:
# print('y:', y, ',predict:', model.predict(X_test).flatten())

In [ ]:
testdata = X_test[201:300,:]
testpred = model.predict(testdata)
testreal = y_test[201:300]
print(testpred, testreal)

In [ ]:
# 교차 검증
val_model = build_model(num_input=15)
history = val_model.fit(X_train, y_train, batch_size=32, epochs=500, validation_split=0.25, verbose=2)

In [ ]:
import matplotlib.pyplot as plt

def plot_loss_curve(total_epoch=10, start=1):
    plt.figure(figsize=(15, 5))
    plt.plot(range(start, total_epoch +1), history.history['loss'][start-1:total_epoch], label='Train')
    plt.plot(range(start, total_epoch +1), history.history['val_loss'][start-1:total_epoch], label='Validation')
    plt.xlabel('Epochs')
    plt.ylabel('mse')
    plt.legend()
    plt.show()

In [ ]:
plot_loss_curve(total_epoch=500, start=1)

In [ ]:
plot_loss_curve(total_epoch=500, start=20)

In [ ]:
from keras.models import load_model
model.save('eta.h5')

In [ ]:
from keras.models import load_model
loaded_model = tf.keras.models.load_model('eta.h5')

In [ ]:
testdata = X_test[101:200,:]
testpred = loaded_model.predict(testdata)
testreal = y_test[101:200]
print(testpred, testreal)

In [ ]:
from sklearn.metrics import mean_absolute_error

predict = loaded_model.predict(X_test)

mae = mean_absolute_error(y_test, predict)
print("mae:", mae)